# String Operations

## Overview

Pandas exposes vectorised string methods via the `.str` accessor, which mirrors Python's built-in `str` methods but operates element-wise on a Series without explicit loops. Missing values are propagated automatically.

**Key `.str` methods:**

| Method | Description |
|---|---|
| `.str.lower()` / `.str.upper()` | Case normalisation |
| `.str.strip()` / `.str.lstrip()` / `.str.rstrip()` | Whitespace removal |
| `.str.replace(pat, repl, regex=True)` | Substitution |
| `.str.contains(pat, regex=True)` | Boolean membership |
| `.str.extract(pat)` | Capture group extraction |
| `.str.split(pat, expand=True)` | Split into columns |
| `.str.startswith()` / `.str.endswith()` | Prefix / suffix matching |

---

In [1]:
import numpy as np
import pandas as pd
import re

rng = np.random.default_rng(42)

# Simulate messy site metadata with common string problems
n = 80
raw = pd.DataFrame({
    'site_code':  [f'  site_{rng.integers(1,21):02d}_2023  ' for _ in range(n)],
    'observer':   rng.choice(['J. Smith', 'j.smith', 'Jane Smith', 'SMITH J', 'A. Jones', 'a.jones'], n),
    'notes':      rng.choice([
        'water clear; flow = 2.3 m/s',
        'turbid water, flow=0.8m/s',
        'Flow: 1.5 m/s - water clear',
        'no flow observed',
        'high turbidity; flow~3.1m/s',
        ''
    ], n),
    'species_list': rng.choice([
        'Salmo trutta, Cottus gobio, Anguilla anguilla',
        'Salmo trutta',
        'Cottus gobio, Lampetra planeri',
        'Anguilla anguilla, Salmo salar'
    ], n)
})
raw.head()

,site_code,observer,notes,species_list
0,site_02_2023,a.jones,,"Salmo trutta, Cottus gobio, Anguilla anguilla"
1,site_16_2023,Jane Smith,no flow observed,"Cottus gobio, Lampetra planeri"
2,site_14_2023,J. Smith,Flow: 1.5 m/s - water clear,"Salmo trutta, Cottus gobio, Anguilla anguilla"
3,site_09_2023,A. Jones,Flow: 1.5 m/s - water clear,"Cottus gobio, Lampetra planeri"
4,site_09_2023,SMITH J,Flow: 1.5 m/s - water clear,"Cottus gobio, Lampetra planeri"


---
## Cleaning and Normalising Strings

In [2]:
df = raw.copy()

# Strip whitespace and normalise case
df['site_code'] = df['site_code'].str.strip().str.lower()
print('Sample site codes:', df['site_code'].unique()[:4])

# Extract structured parts: site number and year from 'site_01_2023'
extracted = df['site_code'].str.extract(r'site_(?P<site_num>\d+)_(?P<year>\d{4})')
df[['site_num', 'year']] = extracted
df['site_num'] = df['site_num'].astype(int)
print(df[['site_code','site_num','year']].head(3))

# Standardise observer names — fuzzy matching is complex; here use explicit map
observer_map = {
    'j. smith': 'J. Smith', 'j.smith': 'J. Smith', 'jane smith': 'J. Smith', 'smith j': 'J. Smith',
    'a. jones': 'A. Jones', 'a.jones': 'A. Jones'
}
df['observer_clean'] = df['observer'].str.lower().str.strip().map(observer_map)
print('Observer counts:', df['observer_clean'].value_counts().to_dict())

Sample site codes: <StringArray>
['site_02_2023', 'site_16_2023', 'site_14_2023', 'site_09_2023']
Length: 4, dtype: str
      site_code  site_num  year
0  site_02_2023         2  2023
1  site_16_2023        16  2023
2  site_14_2023        14  2023
Observer counts: {'J. Smith': 53, 'A. Jones': 27}


---
## Extracting Values from Free-Text Fields

In [3]:
# Extract flow velocity from notes — multiple formats in real data
# Matches: 'flow = 2.3', 'flow=0.8', 'Flow: 1.5', 'flow~3.1'
flow_pattern = r'[Ff]low\s*[=:~]\s*(?P<flow>[\d.]+)'

df['flow_ms'] = (
    df['notes']
    .str.extract(flow_pattern)['flow']
    .astype(float)
)

print(f'Flow extracted from {df["flow_ms"].notna().sum()} of {len(df)} notes')
print(df[['notes','flow_ms']].dropna().head(5))

# Flag water clarity
df['water_clear'] = df['notes'].str.contains(r'\bclear\b', case=False, regex=True, na=False)
df['turbid']      = df['notes'].str.contains(r'\bturbid\b', case=False, regex=True, na=False)

Flow extracted from 63 of 80 notes
                         notes  flow_ms
2  Flow: 1.5 m/s - water clear      1.5
3  Flow: 1.5 m/s - water clear      1.5
4  Flow: 1.5 m/s - water clear      1.5
5  high turbidity; flow~3.1m/s      3.1
6    turbid water, flow=0.8m/s      0.8


---
## Splitting and Expanding

In [4]:
# Split comma-separated species list into rows (tidy format)
species_long = (
    df[['site_num', 'species_list']]
    .assign(species = df['species_list'].str.split(', '))
    .explode('species')
    .drop(columns='species_list')
    .reset_index(drop=True)
)
print('Species in long format:')
print(species_long.head(8))
print(f'\nUnique species: {species_long["species"].nunique()}')
print('Species counts:')
print(species_long['species'].value_counts())

Species in long format:
   site_num            species
0         2       Salmo trutta
1         2       Cottus gobio
2         2  Anguilla anguilla
3        16       Cottus gobio
4        16   Lampetra planeri
5        14       Salmo trutta
6        14       Cottus gobio
7        14  Anguilla anguilla

Unique species: 5
Species counts:
species
Salmo trutta         44
Cottus gobio         42
Anguilla anguilla    36
Lampetra planeri     21
Salmo salar          15
Name: count, dtype: int64


---

## Common Pitfalls

**1. Using `.str` methods on non-string dtype columns**  
Calling `.str.contains()` on an `int64` or `float64` column raises an `AttributeError`. Also, `.str` methods on `category` dtype may behave unexpectedly — convert to `object` first with `.astype(str)` if needed.

**2. Forgetting `na=False` in `.str.contains()`**  
By default, `.str.contains()` returns `NaN` for missing values rather than `False`. This means boolean operations like `df[df['col'].str.contains('pattern')]` will fail or silently exclude NaN rows differently than expected. Always set `na=False` unless you specifically need to preserve NaN.

**3. Writing greedy regex patterns that over-match**  
The pattern `r'flow=([\d.]+)'` will match `'flow=2.3'` correctly but also `'overflow=2.3'`. Use word boundaries (`\b`) and anchors to constrain matches: `r'\bflow\s*=\s*([\d.]+)'`. Always test regex on a representative sample before applying to the full column.

**4. Using `.str.replace()` without `regex=False` for literal strings**  
`.str.replace('(mg/L)', '')` fails because `(` and `)` are regex metacharacters. Either escape them (`r'\(mg/L\)'`) or pass `regex=False` for literal replacements. The default changed in pandas 1.2 — always be explicit.

**5. Not accounting for extra whitespace inside extracted values**  
After `.str.extract()` or `.str.split()`, individual values often have leading/trailing spaces: `' Salmo trutta'`. Always call `.str.strip()` on extracted columns before using them in groupby, merge, or value_counts operations.

---
*python_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*